# 🚀 Pipeline: JSON → CSV
Reads all JSON source files, normalizes them, and saves to CSV.

## 1. Imports

In [11]:
import pandas as pd
import json
import os

print("✅ Imports ready")


✅ Imports ready


## 2. Config — Source Paths & CSV Output

In [12]:
CSV_DIR = r"C:/Projects/Project 1/CSV outputs"

SOURCES = {
    "users":    r"C:/Projects/Project 1/Json data/users/users.json",
    "todos":    r"C:/Projects/Project 1/Json data/todos/todos.json",
    "posts":    r"C:/Projects/Project 1/Json data/posts/posts.json",
    "comments": r"C:/Projects/Project 1/Json data/comments/comments.json",
    "products": r"C:/Projects/Project 1/Json data/products/products.json",
    "recipes":  r"C:/Projects/Project 1/Json data/recipes/recipes.json",
    "quotes":   r"C:/Projects/Project 1/Json data/quotes/quotes.json",
    "carts":    r"C:/Projects/Project 1/Json data/carts/carts.json",
}

print(f"💾 CSV output folder : {CSV_DIR}")
print(f"📦 Sources configured: {len(SOURCES)}")
for name, path in SOURCES.items():
    print(f"   {name:<12} → {path}")


💾 CSV output folder : C:/Projects/Project 1/CSV outputs
📦 Sources configured: 8
   users        → C:/Projects/Project 1/Json data/users/users.json
   todos        → C:/Projects/Project 1/Json data/todos/todos.json
   posts        → C:/Projects/Project 1/Json data/posts/posts.json
   comments     → C:/Projects/Project 1/Json data/comments/comments.json
   products     → C:/Projects/Project 1/Json data/products/products.json
   recipes      → C:/Projects/Project 1/Json data/recipes/recipes.json
   quotes       → C:/Projects/Project 1/Json data/quotes/quotes.json
   carts        → C:/Projects/Project 1/Json data/carts/carts.json


## 3. Helper Function — JSON to CSV

In [13]:
def json_to_csv(name, json_path, csv_dir):
    print(f"\n{'='*50}")
    print(f"🔄 Processing: {name}")
    print(f"{'='*50}")

    # ── Read ──────────────────────────────────────────
    print(f"  📂 Reading  : {json_path}")
    if not os.path.exists(json_path):
        print(f"  ❌ File not found — skipping")
        return {"name": name, "status": "error", "error": "File not found"}

    with open(json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Handle list or dict wrapper
    if isinstance(raw, list):
        records = raw
    elif isinstance(raw, dict):
        records = next((v for v in raw.values() if isinstance(v, list)), [raw])
    else:
        records = [raw]

    print(f"  ✅ Loaded   : {len(records)} records")

    # ── Normalize ─────────────────────────────────────
    df = pd.json_normalize(records)
    df.columns = [col.replace(".", "_") for col in df.columns]
    print(f"  ✅ Normalized: {df.shape[0]} rows × {df.shape[1]} columns")

    # ── Save CSV ──────────────────────────────────────
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = os.path.join(csv_dir, f"{name}.csv")
    df.to_csv(csv_path, index=False)
    size_kb = round(os.path.getsize(csv_path) / 1024, 1)
    print(f"  💾 Saved    : {csv_path} ({size_kb} KB)")

    return {"name": name, "status": "success", "rows": len(df), "cols": len(df.columns), "size_kb": size_kb}

print("✅ Helper function defined")


✅ Helper function defined


## 4. Run Pipeline — Process All Files

In [14]:
print("\n🚀 Starting pipeline...\n")

results = []
for name, json_path in SOURCES.items():
    result = json_to_csv(name, json_path, CSV_DIR)
    results.append(result)

print("\n✅ All files processed")



🚀 Starting pipeline...


🔄 Processing: users
  📂 Reading  : C:/Projects/Project 1/Json data/users/users.json
  ✅ Loaded   : 208 records
  ✅ Normalized: 208 rows × 52 columns
  💾 Saved    : C:/Projects/Project 1/CSV outputs\users.csv (147.0 KB)

🔄 Processing: todos
  📂 Reading  : C:/Projects/Project 1/Json data/todos/todos.json
  ✅ Loaded   : 254 records
  ✅ Normalized: 254 rows × 4 columns
  💾 Saved    : C:/Projects/Project 1/CSV outputs\todos.csv (10.5 KB)

🔄 Processing: posts
  📂 Reading  : C:/Projects/Project 1/Json data/posts/posts.json
  ✅ Loaded   : 251 records
  ✅ Normalized: 251 rows × 8 columns
  💾 Saved    : C:/Projects/Project 1/CSV outputs\posts.csv (87.7 KB)

🔄 Processing: comments
  📂 Reading  : C:/Projects/Project 1/Json data/comments/comments.json
  ✅ Loaded   : 340 records
  ✅ Normalized: 340 rows × 7 columns
  💾 Saved    : C:/Projects/Project 1/CSV outputs\comments.csv (23.7 KB)

🔄 Processing: products
  📂 Reading  : C:/Projects/Project 1/Json data/products/products.

## 5. Summary

In [15]:
print("\n" + "=" * 50)
print("📊 PIPELINE SUMMARY")
print("=" * 50)

success = [r for r in results if r["status"] == "success"]
failed  = [r for r in results if r["status"] == "error"]

print(f"  ✅ Succeeded : {len(success)}")
print(f"  ❌ Failed    : {len(failed)}")
print()

print(f"{'Name':<12} {'Status':<10} {'Rows':<8} {'Cols':<8} {'Size (KB)'}")
print("-" * 50)
for r in results:
    if r["status"] == "success":
        print(f"  ✅ {r['name']:<10} {'success':<10} {r['rows']:<8} {r['cols']:<8} {r['size_kb']}")
    else:
        print(f"  ❌ {r['name']:<10} ERROR — {r.get('error', '')}")

print()
print(f"💾 All CSVs saved to: {CSV_DIR}")



📊 PIPELINE SUMMARY
  ✅ Succeeded : 8
  ❌ Failed    : 0

Name         Status     Rows     Cols     Size (KB)
--------------------------------------------------
  ✅ users      success    208      52       147.0
  ✅ todos      success    254      4        10.5
  ✅ posts      success    251      8        87.7
  ✅ comments   success    340      7        23.7
  ✅ products   success    194      27       237.8
  ✅ recipes    success    50       16       35.8
  ✅ quotes     success    1454     3        147.3
  ✅ carts      success    50       7        50.4

💾 All CSVs saved to: C:/Projects/Project 1/CSV outputs
